# Algorithms 22: Simplex Method**Learning Objectives:**1. Understand Linear Programming and Optimization under Constraints2. Introduce Slack Variables to convert inequalities to equations3. Implement the Simplex Tableau pivoting algorithm**Prerequisites:** Algorithms 15**Estimated time:** 120 minutes**Textbook:** *Justin Skycak — Intro to Algorithms & Machine Learning* Chapter 22

In [ ]:
# Google Colab Setup!pip install -q ipywidgets jupyterquiz ipytest plotly sympy networkx pandas numpy matplotlib tqdmimport json, os, sysfrom pathlib import Pathif Path('/content').exists():    repo_root = Path('/content/upskill')else:    repo_root = Path.cwd()if str(repo_root) not in sys.path:    sys.path.insert(0, str(repo_root))try:    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )except ModuleNotFoundError:    Path('learning_tools.py').write_text('"""Colab-first learning helpers for the interactive math curriculum.\n\nThe functions in this module are intentionally lightweight: they work in a\nplain notebook, in Google Colab, and in local Jupyter without requiring a\nbackend service.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, asdict, field\nfrom datetime import datetime, timedelta, timezone\nimport json\nimport math\nimport os\nimport re\nfrom pathlib import Path\nfrom typing import Any, Callable, Iterable\n\n\nSCHEMA_VERSION = 1\nDEFAULT_PROGRESS_FILENAME = "upskill_math_progress.json"\nDRIVE_PROGRESS_DIR = "MyDrive/upskill"\n\n\ndef _utcnow() -> datetime:\n    return datetime.now(timezone.utc)\n\n\ndef _iso(dt: datetime) -> str:\n    return dt.astimezone(timezone.utc).replace(microsecond=0).isoformat()\n\n\ndef _parse_iso(value: str | None) -> datetime | None:\n    if not value:\n        return None\n    try:\n        return datetime.fromisoformat(value.replace("Z", "+00:00"))\n    except ValueError:\n        return None\n\n\n@dataclass\nclass Skill:\n    """A small curriculum skill descriptor."""\n\n    id: str\n    title: str\n    notebook: str\n    level: int\n    prerequisites: list[str] = field(default_factory=list)\n    tags: list[str] = field(default_factory=list)\n\n    def to_dict(self) -> dict[str, Any]:\n        return asdict(self)\n\n\ndef setup_colab(\n    progress_filename: str = DEFAULT_PROGRESS_FILENAME,\n    mount_drive: bool = True,\n    verbose: bool = True,\n) -> Path:\n    """Prepare a notebook session and return the progress-file path.\n\n    In Colab, this mounts Google Drive when available. Outside Colab, it falls\n    back to the current working directory.\n    """\n\n    progress_path = Path(progress_filename)\n    try:\n        import google.colab  # type: ignore  # noqa: F401\n        in_colab = True\n    except Exception:\n        in_colab = False\n\n    if in_colab and mount_drive:\n        try:\n            from google.colab import drive  # type: ignore\n\n            drive.mount("/content/drive")\n            drive_dir = Path("/content/drive") / DRIVE_PROGRESS_DIR\n            drive_dir.mkdir(parents=True, exist_ok=True)\n            progress_path = drive_dir / progress_filename\n        except Exception as exc:\n            if verbose:\n                print(f"Drive mount skipped; using local progress file. ({exc})")\n\n    if verbose:\n        print(f"Learning environment ready. Progress: {progress_path}")\n    return progress_path\n\n\nclass ProgressStore:\n    """JSON-backed spaced-repetition progress store."""\n\n    def __init__(self, path: str | Path | None = None, learner_id: str = "local"):\n        self.path = Path(path or DEFAULT_PROGRESS_FILENAME)\n        self.learner_id = learner_id\n        self.data = self._load()\n\n    def _default_data(self) -> dict[str, Any]:\n        return {\n            "schema_version": SCHEMA_VERSION,\n            "learner_id": self.learner_id,\n            "skills": {},\n        }\n\n    def _load(self) -> dict[str, Any]:\n        if not self.path.exists():\n            return self._default_data()\n        try:\n            data = json.loads(self.path.read_text(encoding="utf-8"))\n        except Exception:\n            return self._default_data()\n        if data.get("schema_version") != SCHEMA_VERSION:\n            data["schema_version"] = SCHEMA_VERSION\n        data.setdefault("learner_id", self.learner_id)\n        data.setdefault("skills", {})\n        return data\n\n    def save(self) -> None:\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        self.path.write_text(json.dumps(self.data, indent=2), encoding="utf-8")\n\n    def get(self, skill_id: str) -> dict[str, Any]:\n        return self.data.setdefault("skills", {}).setdefault(skill_id, {\n            "attempts": 0,\n            "correct": 0,\n            "confidence": 0,\n            "mastery": 0.0,\n            "last_seen": None,\n            "next_review": None,\n        })\n\n    def record_attempt(\n        self,\n        skill_id: str,\n        correct: bool,\n        confidence: int = 3,\n        item_type: str = "practice",\n    ) -> dict[str, Any]:\n        confidence = max(1, min(5, int(confidence)))\n        row = self.get(skill_id)\n        attempts = int(row.get("attempts", 0)) + 1\n        correct_count = int(row.get("correct", 0)) + int(bool(correct))\n        accuracy = correct_count / attempts\n        confidence_factor = confidence / 5\n        mastery = round((0.75 * accuracy) + (0.25 * confidence_factor), 3)\n\n        row.update({\n            "attempts": attempts,\n            "correct": correct_count,\n            "confidence": confidence,\n            "mastery": mastery,\n            "last_seen": _iso(_utcnow()),\n            "last_item_type": item_type,\n            "next_review": _iso(_utcnow() + review_interval(correct, confidence, attempts)),\n        })\n        self.save()\n        return row\n\n    def review_queue(\n        self,\n        skills: Iterable[Skill | dict[str, Any]] | None = None,\n        limit: int = 8,\n        now: datetime | None = None,\n    ) -> list[dict[str, Any]]:\n        now = now or _utcnow()\n        known_titles = {}\n        if skills:\n            for skill in skills:\n                item = skill.to_dict() if isinstance(skill, Skill) else skill\n                known_titles[item["id"]] = item\n\n        due: list[dict[str, Any]] = []\n        for skill_id, row in self.data.get("skills", {}).items():\n            next_review = _parse_iso(row.get("next_review"))\n            if next_review is None or next_review <= now:\n                due.append({\n                    "id": skill_id,\n                    "title": known_titles.get(skill_id, {}).get("title", skill_id),\n                    "mastery": row.get("mastery", 0.0),\n                    "next_review": row.get("next_review"),\n                    "attempts": row.get("attempts", 0),\n                })\n        due.sort(key=lambda item: (item["mastery"], item["next_review"] or ""))\n        return due[:limit]\n\n    def weak_skills(self, limit: int = 8) -> list[tuple[str, dict[str, Any]]]:\n        rows = list(self.data.get("skills", {}).items())\n        rows.sort(key=lambda kv: (kv[1].get("mastery", 0.0), -kv[1].get("attempts", 0)))\n        return rows[:limit]\n\n\ndef review_interval(correct: bool, confidence: int, attempts: int = 1) -> timedelta:\n    """Return the next review interval from correctness and confidence."""\n\n    if not correct:\n        return timedelta(hours=6) if attempts <= 1 else timedelta(days=1)\n    if confidence <= 2:\n        return timedelta(days=1)\n    if confidence == 3:\n        return timedelta(days=3)\n    high_confidence_days = [7, 14, 30, 60]\n    return timedelta(days=high_confidence_days[min(max(attempts - 1, 0), len(high_confidence_days) - 1)])\n\n\ndef record_attempt(\n    skill_id: str,\n    correct: bool,\n    confidence: int = 3,\n    item_type: str = "practice",\n    store: ProgressStore | None = None,\n) -> dict[str, Any]:\n    store = store or ProgressStore()\n    return store.record_attempt(skill_id, correct, confidence, item_type)\n\n\ndef check(\n    name: str,\n    actual: Any,\n    expected: Any,\n    tolerance: float | None = None,\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Friendly equality/numeric check with optional progress recording."""\n\n    if tolerance is not None and isinstance(actual, (int, float)) and isinstance(expected, (int, float)):\n        ok = math.isclose(actual, expected, rel_tol=tolerance, abs_tol=tolerance)\n    else:\n        ok = actual == expected\n\n    if ok:\n        print(f"PASS: {name}")\n    else:\n        print(f"TRY AGAIN: {name}")\n        print(f"  expected: {expected!r}")\n        print(f"  actual:   {actual!r}")\n\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "check", store)\n    return ok\n\n\ndef code_check(\n    name: str,\n    fn: Callable[..., Any],\n    cases: Iterable[tuple[tuple[Any, ...], Any] | tuple[tuple[Any, ...], dict[str, Any], Any]],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Run multiple function cases and report the first failure."""\n\n    all_ok = True\n    for index, case in enumerate(cases, 1):\n        if len(case) == 2:\n            args, expected = case  # type: ignore[misc]\n            kwargs = {}\n        else:\n            args, kwargs, expected = case  # type: ignore[misc]\n        try:\n            actual = fn(*args, **kwargs)\n            ok = actual == expected\n        except Exception as exc:\n            actual = f"{type(exc).__name__}: {exc}"\n            ok = False\n        if not ok:\n            all_ok = False\n            print(f"TRY AGAIN: {name} case {index}")\n            print(f"  args:     {args}")\n            print(f"  expected: {expected!r}")\n            print(f"  actual:   {actual!r}")\n            break\n    if all_ok:\n        print(f"PASS: {name} ({index} cases)")\n    if skill_id:\n        record_attempt(skill_id, all_ok, confidence, "code_check", store)\n    return all_ok\n\n\ndef normalize_answer(value: Any) -> str:\n    text = str(value).strip().lower()\n    text = re.sub(r"\\s+", " ", text)\n    text = re.sub(r"[^a-z0-9.+/=\\- ]", "", text)\n    text = re.sub(r"\\s*=\\s*", "=", text)\n    return text\n\n\ndef short_answer_check(\n    name: str,\n    answer: Any,\n    accepted: Iterable[Any],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    normalized = normalize_answer(answer)\n    accepted_norm = {normalize_answer(item) for item in accepted}\n    ok = normalized in accepted_norm\n    print(f"{\'PASS\' if ok else \'TRY AGAIN\'}: {name}")\n    if not ok:\n        print("  Re-read the prompt and try to retrieve the answer before opening a hint.")\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "short_answer", store)\n    return ok\n\n\ndef hint_box(title: str, hints: list[str], unlock: bool = False) -> None:\n    """Display staged hints. Full hints require unlock=True."""\n\n    print(title)\n    if not hints:\n        print("No hints for this item.")\n        return\n    if not unlock:\n        print(f"Hint 1: {hints[0]}")\n        print("Run again with unlock=True after a real attempt to see more.")\n        return\n    for index, hint in enumerate(hints, 1):\n        print(f"Hint {index}: {hint}")\n\n\ndef mastery_dashboard(\n    store: ProgressStore | None = None,\n    skills: Iterable[Skill | dict[str, Any]] | None = None,\n    next_notebook: str | None = None,\n) -> None:\n    store = store or ProgressStore()\n    due = store.review_queue(skills=skills)\n    weak = store.weak_skills()\n    total = len(store.data.get("skills", {}))\n    mastered = sum(1 for row in store.data.get("skills", {}).values() if row.get("mastery", 0) >= 0.8)\n\n    print("Mastery Dashboard")\n    print("=================")\n    print(f"Skills attempted: {total}")\n    print(f"Skills at mastery >= 0.80: {mastered}")\n    if due:\n        print("\\nDue for review:")\n        for item in due:\n            print(f"- {item[\'title\']} (mastery {item[\'mastery\']})")\n    else:\n        print("\\nNo due reviews yet.")\n    if weak:\n        print("\\nWeakest skills:")\n        for skill_id, row in weak:\n            print(f"- {skill_id}: mastery {row.get(\'mastery\', 0)} after {row.get(\'attempts\', 0)} attempts")\n    if next_notebook:\n        print(f"\\nRecommended next notebook: {next_notebook}")\n', encoding='utf-8')    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )progress_path = setup_colab()store = ProgressStore(progress_path)print('Ready for retrieval practice.')

In [ ]:
NOTEBOOK = {    "id": "22_simplex_method",    "level": 3,    "title": "Algorithms 22: Simplex Method",    "prerequisites": [        "Algorithms 15"    ],    "skills_taught": [        "lesson.algorithms_22_simplex_method"    ],    "skills_practiced": [        "py.arithmetic.expressions"    ],    "next_notebook": "23-25_regression_pseudoinverse.ipynb"}SKILLS = [    {        "id": "lesson.algorithms_22_simplex_method",        "title": "Lesson Algorithms 22 Simplex Method",        "notebook": "22_simplex_method.ipynb",        "level": 3    },    {        "id": "py.arithmetic.expressions",        "title": "Py Arithmetic Expressions",        "notebook": "22_simplex_method.ipynb",        "level": 3    }]

## Before You Learn: Retrieval GateClose notes for a moment. Try to pull prerequisite ideas from memory before continuing. Getting stuck is useful data for the spaced-review system.

In [ ]:
due = store.review_queue(skills=SKILLS, limit=5)if due:    print('Due review items:')    for item in due:        print(f"- {item['title']} (mastery {item['mastery']})")else:    print('No due reviews yet. Use the recall prompt below to warm up.')

**Recall prompt.** Before reading, name one key idea or procedure you remember from: Algorithms 15.

In [ ]:
recall_answer = ''  # write a one-sentence memory before continuingretrieved = len(recall_answer.strip()) >= 12record = store.record_attempt('lesson.algorithms_22_simplex_method', retrieved, confidence=3, item_type='prerequisite_recall')print('Recorded retrieval attempt.' if retrieved else 'Write a real memory first, then rerun this cell.')record

## Learning LoopUse the existing lesson cells below as the micro-lesson and practice surface. For each exercise: predict first, attempt from memory, run checks, then open explanations only after the attempt.

In [ ]:
# Google Colab Setupimport mathimport matplotlib.pyplot as pltprint('Ready ✅')

---## Phase −1 — Theoretical Foundation### Maximizing Profit Under Constraints> 📖 *From Algorithms Ch. 22:* The simplex method maximizes linear expressions subject to linear constraints. e.g., A business determines the most profitable inventory of products to produce given limited raw materials.Maximize: $M = x_1 + 2x_2 + x_3$Subject to:$2x_1 + x_2 + x_3 \le 14$$4x_1 + 2x_2 + 3x_3 \le 28$$2x_1 + 5x_2 + 5x_3 \le 30$### Step 1: Slack VariablesConvert inequalities into equations by adding non-negative "slack variables" ($x_4, x_5, x_6$) representing leftover raw materials.$x_4 = 14 - 2x_1 - x_2 - x_3$$x_5 = 28 - 4x_1 - 2x_2 - 3x_3$$x_6 = 30 - 2x_1 - 5x_2 - 5x_3$### Step 2: Basic vs Non-BasicThe variables on the RHS ($x_1, x_2, x_3$) are **non-basic**. We set them to 0. The LHS variables ($x_4, x_5, x_6$) are **basic**. Their values are simply the constants ($14, 28, 30$).If we set $x_1=x_2=x_3=0$, the Profit is 0. We can do better.### Step 3: Pivoting1. Pick the non-basic variable in the Profit equation with the largest positive coefficient ($x_2$ has coefficient 2). We want to increase $x_2$!2. How much can we increase $x_2$? Look at the constraints:   - $x_4 = 14 - x_2 \implies x_2 \le 14$   - $x_5 = 28 - 2x_2 \implies x_2 \le 14$   - $x_6 = 30 - 5x_2 \implies x_2 \le 6$ (The strictest constraint!)3. Since $x_6$ is the bottleneck, we **trade** $x_2$ for $x_6$. We solve the $x_6$ equation for $x_2$, and substitute that expression into *all* other equations (including the Profit equation). 4. Repeat until the Profit equation has no positive coefficients left!

### Comprehension Check ✅Why do we stop when all coefficients in the objective (Profit) equation are negative?<details><summary>💡 Explanation after a retrieval attempt</summary>Because the non-basic variables are locked at 0 (and can never be negative). If the Profit equation is $M = 50 - 2x_1 - 3x_4$, increasing $x_1$ or $x_4$ will only *decrease* the profit from 50. Therefore, 50 is the absolute mathematical maximum.</details>

---## Phase 0 — Math Foundation Practice### 🎯 Your Turn: Algebraic SubstitutionEquation 1: $x_6 = 30 - 2x_1 - 5x_2 - 5x_3$Solve this algebraically for $x_2$. *(Work it out on paper)*<details><summary>💡 Solution</summary>$5x_2 = 30 - 2x_1 - 5x_3 - x_6$$x_2 = 6 - 0.4x_1 - 1x_3 - 0.2x_6$</details>

---## Phase 1 — Algorithm Construction### Step 1: The Tableau RepresentationWriting variables is tedious. We use a Tableau (a 2D array or Dictionary). Instead of strings, we can represent equations as dictionaries mapping variable indices to coefficients. Let's build a `Simplex` class.

In [ ]:
class Simplex:    def __init__(self, objective, constraints):        # objective is a dict mapping var_idx -> coefficient, e.g. {1: 1, 2: 2, 3: 1}        # constraints is a list of tuples (constant, dict_of_coeffs)        # e.g. (14, {1: -2, 2: -1, 3: -1}) representing x4 = 14 - 2x1 - x2 - x3                self.obj = objective        self.eqs = {}                # Create slack variables        slack_idx = max(self.obj.keys()) + 1        for constant, coeffs in constraints:            self.eqs[slack_idx] = {'const': constant, 'coeffs': coeffs}            slack_idx += 1                def get_entering_variable(self):        # Find the var_idx in self.obj with the maximum positive coefficient        # Return None if no positive coefficients remain        best_var = None        max_coeff = 0        # YOUR CODE HERE        return best_var

### Step 2: Finding the Leaving Variable (The Bottleneck)For the chosen entering variable, figure out which basic variable's equation places the strictest limit on it.

In [ ]:
# Add to Simplex class:# def get_leaving_variable(self, enter_var):#     # Look at all equations in self.eqs.#     # If the coefficient of enter_var is negative, it places an upper bound.#     # bound = const / abs(coeff)#     # Find the eq_idx with the smallest non-negative bound.#     best_eq_idx = None#     min_bound = float('inf')#     # YOUR CODE HERE#     return best_eq_idx

### Step 3: PivotingExecute the algebra! 1. Isolate the entering variable in the leaving equation.2. Substitute this new expression into the Objective function, and into *all other* constraint equations.

In [ ]:
# Add to Simplex class:# def pivot(self, enter_var, leave_var):#     # YOUR CODE HERE#     # (This requires careful dictionary manipulation!)#     pass# def solve(self):#     while True:#         enter_var = self.get_entering_variable()#         if enter_var is None: break # Optimal!#         leave_var = self.get_leaving_variable(enter_var)#         self.pivot(enter_var, leave_var)#     return self.obj.get('const', 0) # Maximum profit

---## Phase 2 — Experimental Verification### Solving the Widget FactoryLet's run the example from Phase -1 through your solver. (If you haven't fully coded the `pivot` logic, try doing the dictionary substitutions manually!)

In [ ]:
# obj = {1: 1, 2: 2, 3: 1}# c1 = (14, {1: -2, 2: -1, 3: -1})# c2 = (28, {1: -4, 2: -2, 3: -3})# c3 = (30, {1: -2, 2: -5, 3: -5})# s = Simplex(obj, [c1, c2, c3])# max_profit = s.solve()# print("Maximum Profit:", max_profit) # Should be 12.0!

---## Phase 3 — Olympiad Extension### Challenge: Matrix RepresentationThe dictionary approach is slow. Professional Simplex solvers use a giant 2D matrix (the Simplex Tableau), and perform pivots using exact same logic as your **Reduced Row Echelon Form** (RREF) from Algorithms 15!Combine your RREF matrix skills to implement `MatrixSimplex`.### Chalkboard DefenseLinear Programming is solvable in Polynomial Time. Integer Linear Programming (where you can only produce *whole* quantities of objects, e.g. you can't build 0.5 of a car) is NP-Hard. Why does restricting the variables to integers make the problem exponentially harder to solve?<details><summary>💡 Sketch after a retrieval attempt</summary>The Simplex method glides perfectly along the edges and vertices of a continuous geometric shape (a polytope). If variables must be integers, the geometric vertices are no longer guaranteed to be valid answers! The optimal integer answer might be trapped somewhere deep inside the shape. You essentially have to start brute-forcing integer coordinates near the continuous maximum, leading to exponential branching (Branch and Bound algorithms).</details>---📚 **Next:** Algorithms 23 (Introduction to Regression)

## End-of-Notebook ReviewRetrieve the core idea one more time before leaving. This final recall makes the next review easier.

In [ ]:
final_recall = ''  # summarize the most important idea in your own wordsconfidence = 3  # set 1-5 after answeringok = len(final_recall.strip()) >= 20store.record_attempt('lesson.algorithms_22_simplex_method', ok, confidence=confidence, item_type='end_review')mastery_dashboard(store=store, skills=SKILLS, next_notebook='23-25_regression_pseudoinverse.ipynb')